In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tools import bar, line
from pathlib import Path

FIG_DIR_BNB = Path(r'..\reports\figures\airbnb')
FIG_DIR_OCC = Path(r'..\reports\figures\occupation')
FIG_WIDTH, FIG_HEIGHT = 1600, 500

In [3]:
df_bnb = pd.read_csv(r"..\data\processed\airbnb.csv", sep=';', decimal=',', parse_dates=['mois'])
df_tx = pd.read_csv(r"..\data\processed\tx_occupation.csv", sep=';', decimal=',', parse_dates=['mois'])

---
# Evolution Consultations

In [4]:
fig = line(df_bnb, x='mois', y='consultations', titre='Consultations mensuelles Airbnb', palette=px.colors.qualitative.D3)
fig.update_traces(fill='tozeroy', fillcolor='rgba(33, 150, 243, 0.15)')
fig.update_layout(yaxis=dict(title='Consultations'))

fig.show()
fig.write_image(FIG_DIR_BNB / "evolution_consultations.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Consultations vs Réservations

In [5]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_bnb['mois'],
    y=df_bnb['consultations'],
    name="Consultations",
    mode='lines+markers',
    yaxis="y2",
    line=dict(color='#2596be'),
    hovertemplate='%{x} : %{y:,.0f}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=df_bnb['mois'],
    y=df_bnb['réservations'],
    name="Réservations",
    marker=dict(color='#ff8f2c', opacity=0.9),
    width=1000 * 3600 * 24 * 10,
    hovertemplate='%{x} : %{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title='Consultations vs Réservations mensuelles', 
    template='plotly_white', 
    height=500, 
    hovermode='x unified', 
    separators=". ",
    yaxis=dict(title="Réservations", showgrid=False),
    xaxis=dict(tickformat="%B %Y"),
    yaxis2=dict(
        side='right',
        overlaying='y',
        title="Consultations"
    ),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    )

fig.show()
fig.write_image(FIG_DIR_BNB / "consultations_vs_reservations.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Taux de Conversion

In [6]:
couleurs = ['green' if x > df_bnb['taux_de_réservation'].mean() else 'red' for x in df_bnb['taux_de_réservation']]

fig = bar(df_bnb, x='mois', y='taux_de_réservation', text_auto=False, titre='Taux de conversion mensuel Airbnb', palette=couleurs)

fig.add_hline(
    y=df_bnb['taux_de_réservation'].mean(),
    line_dash='dash',
    line_color='black',
    annotation_text=f"Moyenne : {df_bnb['taux_de_réservation'].mean():.1f}%",
    annotation_position='top right'
)

fig.update_layout(yaxis=dict(ticksuffix=' %'))

fig.show()
fig.write_image(FIG_DIR_BNB / "tx_conversion.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Comparaison 2024 vs 2025 (mois communs juil → déc)

In [7]:
mois_commun = df_bnb[df_bnb['mois'].dt.month.isin(range(7, 13))].copy()
mois_commun['period_label'] = mois_commun['mois'].dt.month_name(locale='fr_FR')

## Consultations

In [8]:
fig = bar(mois_commun, x='period_label', y='consultations', color='année', titre='Consultations juil–déc : 2024 vs 2025', palette=px.colors.qualitative.D3)
fig.update_layout(yaxis=dict(title='Consultations'))

fig.show()
fig.write_image(FIG_DIR_BNB / "consult_2024vs2025.png", width=FIG_WIDTH, height=FIG_HEIGHT)

## Réservations

In [9]:
fig = bar(mois_commun, x='period_label', y='réservations', color='année', titre='Réservations juil–déc : 2024 vs 2025', palette=px.colors.qualitative.D3)
fig.update_layout(yaxis=dict(title='Réservations'))

fig.show()
fig.write_image(FIG_DIR_BNB / "resa_2024vs2025.png", width=FIG_WIDTH, height=FIG_HEIGHT)

### Groupé

In [10]:
fig_consult = bar(mois_commun, x='period_label', y='consultations', color='année', titre='Consultations juil–déc : 2024 vs 2025', palette=px.colors.qualitative.D3)
fig_resa = bar(mois_commun, x='period_label', y='réservations', color='année', titre='Réservations juil–déc : 2024 vs 2025', palette=px.colors.qualitative.D3)

subplot = make_subplots(rows=1, cols=2, subplot_titles=('Consultations', 'Réservations'))
for trace in fig_consult.data:
    subplot.add_trace(trace, row=1, col=1)
for trace in fig_resa.data:
    trace.showlegend = False
    subplot.add_trace(trace, row=1, col=2)

subplot.update_layout(
    title='Comparaison saisonnière Airbnb — mois communs',
    template='plotly_white', 
    height=500,
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    hovermode='x unified', 
    separators=". ",
    )

subplot.show()
subplot.write_image(FIG_DIR_BNB / "consult_resa_2024vs2025.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Taux d'occupation + Prix moyen nuitée (2024)

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_tx['mois'],
    y=df_tx['prix_moyen_nuitée'],
    name="Prix Moyen Nuitée (€)",
    mode='lines+markers',
    yaxis="y2",
    line=dict(color='#2596be'),
    hovertemplate='%{x} : %{y:,.0f}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=df_tx['mois'],
    y=df_tx['taux_occupation'] * 100,
    name="Taux d'occupation (%)",
    marker=dict(color='#ff8f2c', opacity=0.9),
    width=1000 * 3600 * 24 * 10,
    hovertemplate='%{x} : %{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title='Taux d\'occupation et prix moyen par nuitée — 2024', 
    template='plotly_white', 
    height=500, 
    hovermode='x unified', 
    separators=". ",
    yaxis=dict(title="Taux d'occupation (%)", showgrid=False),
    xaxis=dict(tickformat="%B %Y"),
    yaxis2=dict(
        side='right',
        overlaying='y',
        title="Prix Moyen Nuitée (€)"
    ),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    )

fig.show()
fig.write_image(FIG_DIR_OCC / "tx_occup+pxmoy_nuit.png", width=FIG_WIDTH, height=FIG_HEIGHT)